# RTR_Lite_MobileNet — Plant Disease Classification

**Paper:** *RTR_Lite_MobileNetV2: A lightweight and efficient model for plant disease detection and classification*  
Duhan et al., Current Plant Biology 42 (2025) 100459

| Hyperparameter | Value | Source |
|---|---|---|
| Framework | PyTorch | Section 3.3 |
| Input size | 224×224 | Section 3.1 |
| Batch size | 32 | Section 3.3 |
| Epochs | 30 | Section 3.3 |
| Optimizer | Adam, LR=0.001 | Section 3.3 |
| LR schedule | ×0.9 every 5 epochs | Section 3.3 |
| Loss | CrossEntropyLoss | Section 3.3 |
| Transfer learning | MobileNetV2 ImageNet1K_V1 | Section 4 |
| Dropout | 0.2 | Standard MobileNetV2 |
| Total params | ~1,050,547 | Section 2.5 |

**Architecture modifications on MobileNetV2:**
- After `features[0]`: + TripletAttention + RES\_Block
- Keep `features[1..15]`, drop `features[16,17,18]`
- In place of dropped layers: + RES\_Block + TripletAttention + RES\_Block
- Classifier: AdaptiveAvgPool → Dropout(0.2) → Linear(160, NUM\_CLASSES)

**2 Scenarios:**
- Scenario 1 (Baseline): Resize + Normalize only
- Scenario 2 (Augmented): Rotation + ColorJitter + HFlip + RandomAffine


In [ ]:
# ── Install dependencies ────────────────────────────────────────────────────
# torch/torchvision/sklearn/matplotlib/seaborn already on Kaggle
!pip install -q torchinfo openpyxl Pillow
print("Dependencies ready.")

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import os, gc, math, warnings
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchinfo import summary as torchinfo_summary
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, recall_score, accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('Imports OK | seed=42 | cudnn.deterministic=True')

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────

# Dataset paths — UPDATE dataset name
DATASET_PATH = '/kaggle/input/YOUR-DATASET-NAME'   # ← update
TRAIN_DIR    = os.path.join(DATASET_PATH, 'train')
VAL_DIR      = os.path.join(DATASET_PATH, 'val')
TEST_DIR     = os.path.join(DATASET_PATH, 'test')

# Hyperparameters (from paper)
IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 30
LR           = 1e-3
LR_STEP      = 5       # decay every 5 epochs
LR_GAMMA     = 0.9     # multiply LR by 0.9
DROPOUT      = 0.2
PRETRAINED   = True    # use ImageNet1K_V1 weights

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Device: {DEVICE} | GPUs available: {n_gpus}')

In [ ]:
# ── Dataset Exploration — auto-detect classes ───────────────────────────────
class_names = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d))
])
NUM_CLASSES = len(class_names)
print(f'NUM_CLASSES = {NUM_CLASSES}')
print(f'Classes: {class_names}')

for split, d in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    if os.path.exists(d):
        n = sum(
            len(os.listdir(os.path.join(d, c)))
            for c in os.listdir(d)
            if os.path.isdir(os.path.join(d, c))
        )
        print(f'  {split:5s}: {n:6d} images')
    else:
        print(f'  {split:5s}: NOT FOUND -> {d}')


In [ ]:
# ── Dataset Distribution — imbalance check + auto class_weights ─────────────
import numpy as np

_tmp_ds = ImageFolder(TRAIN_DIR)
class_names = _tmp_ds.classes          # authoritative order from ImageFolder
NUM_CLASSES  = len(class_names)

# Count images per class
counts = np.zeros(NUM_CLASSES, dtype=int)
for _, label in _tmp_ds.samples:
    counts[label] += 1

# Bar chart
fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES * 0.8), 4))
ax.bar(class_names, counts, color='steelblue', edgecolor='white')
ax.set_title('Training Set — Class Distribution', fontweight='bold')
ax.set_xlabel('Class'); ax.set_ylabel('# Images')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'NUM_CLASSES = {NUM_CLASSES}')
for n, c in zip(class_names, counts):
    print(f'  {n}: {c}')

# Imbalance check — Rule 11: if max/min >= 1.5, use class_weights
ratio = counts.max() / counts.min()
print(f'\nImbalance ratio (max/min) = {ratio:.2f}')
if ratio >= 1.5:
    labels_array = np.array([lbl for _, lbl in _tmp_ds.samples])
    cw = compute_class_weight('balanced',
                               classes=np.arange(NUM_CLASSES),
                               y=labels_array)
    CLASS_WEIGHTS = torch.tensor(cw, dtype=torch.float32)
    print(f'[Imbalanced] Class weights applied: {CLASS_WEIGHTS.numpy()}')
else:
    CLASS_WEIGHTS = None
    print('[Balanced] No class weighting needed.')

del _tmp_ds

In [ ]:
# ── RTR_Lite_MobileNet Architecture ─────────────────────────────────────────
# Ref: Duhan et al. (2025). Current Plant Biology.
# Modified MobileNetV2 with Triplet Attention + RES_Block (SENet + ECA)

# ─── Modified SENet ──────────────────────────────────────────────────────────
class ModifiedSENet(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        rd = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, rd, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(rd, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c = x.shape[:2]
        avg = self.avg_pool(x).view(b, c)
        mxp = self.max_pool(x).view(b, c)
        w = self.fc(avg + mxp).view(b, c, 1, 1)
        return x * w


# ─── ECA Module ──────────────────────────────────────────────────────────────
class ECA(nn.Module):
    def __init__(self, channels, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv     = nn.Conv1d(1, 1, kernel_size=k_size,
                                  padding=k_size // 2, bias=False)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = y.transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)


# ─── RES_Block (Eq. 1: x ⊕ ECA(SENet(x))) ───────────────────────────────────
class RES_Block(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.se  = ModifiedSENet(channels)
        self.eca = ECA(channels)

    def forward(self, x):
        return x + self.eca(self.se(x))   # residual skip — Eq.(1) paper


# ─── Triplet Attention ───────────────────────────────────────────────────────
class _ZPool(nn.Module):
    def forward(self, x):
        return torch.cat(
            [x.max(dim=1, keepdim=True)[0],
             x.mean(dim=1, keepdim=True)], dim=1
        )


class _AttGate(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.zpool = _ZPool()
        self.conv  = nn.Conv2d(2, 1, kernel_size=kernel_size,
                               padding=pad, bias=False)
        self.bn    = nn.BatchNorm2d(1)

    def forward(self, x):
        attn = torch.sigmoid(self.bn(self.conv(self.zpool(x))))
        return x * attn


class TripletAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.hc = _AttGate(kernel_size)
        self.wc = _AttGate(kernel_size)
        self.hw = _AttGate(kernel_size)

    def forward(self, x):
        xp1 = x.permute(0, 2, 1, 3).contiguous()
        y1  = self.hc(xp1).permute(0, 2, 1, 3).contiguous()
        xp2 = x.permute(0, 3, 1, 2).contiguous()
        y2  = self.wc(xp2).permute(0, 2, 3, 1).contiguous()
        y3  = self.hw(x)
        return (y1 + y2 + y3) / 3.0


# ─── RTR_Lite_MobileNet ──────────────────────────────────────────────────────
class RTR_Lite_MobileNet(nn.Module):
    '''
    RTR_Lite_MobileNet: Attention-fused Residual Lite MobileNetV2.
    Modifications (Section 2.5, Fig 5b):
      - After features[0]: TripletAttention + RES_Block
      - features[1..15] kept intact (160 ch output)
      - features[16,17,18] replaced by:
          (RES_Block + TripletAttention + RES_Block) × 2  [repeated twice]
      - Classifier: AdaptiveAvgPool -> Dropout(0.2) -> Linear(160, num_classes)
    Target params: ~1,050,547 (Duhan et al. 2025, ablation Model 6)
    '''
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        weights = (
            torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1
            if pretrained else None
        )
        base = torchvision.models.mobilenet_v2(weights=weights)

        self.first_conv = base.features[0]      # (B, 32, 112, 112)

        # Attention after first conv (32 channels)
        self.triplet1 = TripletAttention(kernel_size=7)
        self.res1     = RES_Block(32)

        # Backbone: features[1..15] → 160 channels
        self.backbone = nn.Sequential(*list(base.features[1:16]))

        # Replacement for features[16,17,18] — repeated TWICE (Section 2.5)
        # First repetition
        self.res2     = RES_Block(160)
        self.triplet2 = TripletAttention(kernel_size=7)
        self.res3     = RES_Block(160)
        # Second repetition
        self.res4     = RES_Block(160)
        self.triplet3 = TripletAttention(kernel_size=7)
        self.res5     = RES_Block(160)

        # Classifier head
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.2)
        self.fc   = nn.Linear(160, num_classes)

    def forward(self, x):
        x = self.first_conv(x)    # (B, 32, 112, 112)
        x = self.triplet1(x)
        x = self.res1(x)
        x = self.backbone(x)      # (B, 160, 7, 7)
        # First repetition
        x = self.res2(x)
        x = self.triplet2(x)
        x = self.res3(x)
        # Second repetition
        x = self.res4(x)
        x = self.triplet3(x)
        x = self.res5(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)


print('RTR_Lite_MobileNet architecture loaded.')

In [ ]:
# ── Model Summary (torchinfo) ────────────────────────────────────────────────
_tmp = RTR_Lite_MobileNet(num_classes=NUM_CLASSES,
                           pretrained=PRETRAINED).to(DEVICE)
_m = _tmp.module if isinstance(_tmp, nn.DataParallel) else _tmp
torchinfo_summary(
    _m,
    input_size=(1, 3, IMG_SIZE, IMG_SIZE),
    col_names=['input_size', 'output_size', 'num_params'],
    depth=4,
    device=DEVICE,
    verbose=1,
)
del _tmp, _m

In [ ]:
# ── Training & Evaluation Functions ─────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    preds_all, labels_all = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out   = model(imgs)
        loss  = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        preds       = out.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, preds_all, labels_all


def train_model(model, train_loader, val_loader,
                optimizer, scheduler, criterion,
                device, epochs, ckpt_path):
    history = {'train_loss':[], 'val_loss':[],
               'train_acc':[],  'val_acc':[]}
    best_val_acc = 0.0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc       = train_one_epoch(
            model, train_loader, optimizer, criterion, device)
        vl_loss, vl_acc, _, _ = evaluate(
            model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        print(f'Epoch {epoch:3d}/{epochs} | '
              f'Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | '
              f'Val   Loss={vl_loss:.4f} Acc={vl_acc:.4f}')

        if vl_acc > best_val_acc:          # checkpoint by val_accuracy (Rule 10)
            best_val_acc = vl_acc
            _state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(_state, ckpt_path)

    # Reload best checkpoint before test evaluation
    _target = model.module if isinstance(model, nn.DataParallel) else model
    _target.load_state_dict(torch.load(ckpt_path, map_location=device))
    return history


def count_params(model):
    m = model.module if isinstance(model, nn.DataParallel) else model
    return sum(p.numel() for p in m.parameters())


print('Functions loaded.')

---
## Scenario 1 — Baseline (No Augmentation)
Transform: **Resize(224) → ToTensor → Normalize** only.


In [ ]:
# ── Scenario 1: No-Aug DataLoaders ─────────────────────────────────────────
tf_noaug = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

train_ds1 = ImageFolder(TRAIN_DIR, transform=tf_noaug)
val_ds1   = ImageFolder(VAL_DIR,   transform=tf_noaug)
test_ds1  = ImageFolder(TEST_DIR,  transform=tf_noaug)

assert train_ds1.classes == class_names, 'Class order mismatch!'

train_loader1 = DataLoader(train_ds1, batch_size=BATCH_SIZE,
                            shuffle=True,  num_workers=4, pin_memory=True)
val_loader1   = DataLoader(val_ds1,   batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=4, pin_memory=True)
test_loader1  = DataLoader(test_ds1,  batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=4, pin_memory=True)

print(f'S1 ready — Train:{len(train_ds1)}, Val:{len(val_ds1)}, Test:{len(test_ds1)}')

In [ ]:
# ── Scenario 1: Initialize + Train ─────────────────────────────────────────
model_s1 = RTR_Lite_MobileNet(NUM_CLASSES, pretrained=PRETRAINED).to(DEVICE)
# DataParallel skipped: TripletAttention permute ops incompatible with DP scatter

optimizer_s1  = torch.optim.Adam(model_s1.parameters(), lr=LR)
scheduler_s1  = torch.optim.lr_scheduler.StepLR(
                    optimizer_s1, step_size=LR_STEP, gamma=LR_GAMMA)

# Class weighting (auto from distribution cell)
if CLASS_WEIGHTS is not None:
    criterion_s1 = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))
    print('[S1] CrossEntropyLoss with class_weights')
else:
    criterion_s1 = nn.CrossEntropyLoss()
    print('[S1] CrossEntropyLoss (no weighting)')

print('=== Scenario 1 — Baseline (No Augmentation) ===')
print(f'Total parameters: {count_params(model_s1):,}')

history_s1 = train_model(
    model_s1, train_loader1, val_loader1,
    optimizer_s1, scheduler_s1, criterion_s1,
    DEVICE, EPOCHS,
    ckpt_path='/kaggle/working/best_model_s1.pth',
)
print('Training complete.')

In [ ]:
# ── Scenario 1: Classification Report ──────────────────────────────────────
_, s1_acc, s1_preds, s1_labels = evaluate(
    model_s1, test_loader1, criterion_s1, DEVICE)

s1_f1  = f1_score(s1_labels, s1_preds, average='macro')
s1_rec = recall_score(s1_labels, s1_preds, average='macro')

print('\n=== Scenario 1 — Test Metrics ===')
print(f'Accuracy     : {s1_acc:.4f}')
print(f'F1-Score (M) : {s1_f1:.4f}')
print(f'Recall   (M) : {s1_rec:.4f}')
print(f'Parameters   : {count_params(model_s1):,}')
print('\n--- Per-class Classification Report ---')
print(classification_report(s1_labels, s1_preds,
      target_names=class_names, digits=4))


In [ ]:
# ── Scenario 1: Confusion Matrix ────────────────────────────────────────────
BASELINE_CM_PATH = '/kaggle/working/baseline_confusion_matrix.png'

cm1 = confusion_matrix(s1_labels, s1_preds)
fig, ax = plt.subplots(figsize=(max(6, NUM_CLASSES), max(5, NUM_CLASSES - 1)))
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix — Scenario 1 (Baseline)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True',      fontsize=12)
plt.tight_layout()
plt.savefig(BASELINE_CM_PATH, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {BASELINE_CM_PATH}')


In [ ]:
# ── Scenario 1: Learning Curve ──────────────────────────────────────────────
BASELINE_LC_PATH = '/kaggle/working/baseline_learning_curve.png'

ep = range(1, len(history_s1['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep, history_s1['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(ep, history_s1['val_loss'],   label='Val Loss',   color='tomato')
axes[0].set_title('Loss — Scenario 1 (Baseline)', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history_s1['train_acc'], label='Train Acc', color='steelblue')
axes[1].plot(ep, history_s1['val_acc'],   label='Val Acc',   color='tomato')
axes[1].set_title('Accuracy — Scenario 1 (Baseline)', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('RTR_Lite_MobileNet | Scenario 1 — No Augmentation',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(BASELINE_LC_PATH, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {BASELINE_LC_PATH}')


In [ ]:
# ── Scenario 1: metrics dict (for Excel export) ─────────────────────────────
baseline_metrics = {
    'model_name'   : 'RTR_Lite_MobileNet',
    'accuracy'     : s1_acc,
    'f1_macro'     : s1_f1,
    'recall_macro' : s1_rec,
    'total_params' : count_params(model_s1),
}
print('baseline_metrics:', baseline_metrics)


---
## Scenario 2 — Augmented (4 Techniques)
From paper Section 3.1: **Random Rotation (±30°) · ColorJitter · Horizontal Flip · RandomAffine (scale + shear + translate)**


In [ ]:
# ── Scenario 2: Augmented DataLoaders ──────────────────────────────────────
tf_aug = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomRotation(degrees=30),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1),
                   scale=(0.85, 1.15), shear=10),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

tf_val = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

train_ds2 = ImageFolder(TRAIN_DIR, transform=tf_aug)
val_ds2   = ImageFolder(VAL_DIR,   transform=tf_val)
test_ds2  = ImageFolder(TEST_DIR,  transform=tf_val)

train_loader2 = DataLoader(train_ds2, batch_size=BATCH_SIZE,
                            shuffle=True,  num_workers=4, pin_memory=True)
val_loader2   = DataLoader(val_ds2,   batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=4, pin_memory=True)
test_loader2  = DataLoader(test_ds2,  batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=4, pin_memory=True)

print(f'S2 ready — Train:{len(train_ds2)}, Val:{len(val_ds2)}, Test:{len(test_ds2)}')

In [ ]:
# ── Scenario 2: Re-initialize Model + Train (fresh weights, no data leakage) ─
model_s2 = RTR_Lite_MobileNet(NUM_CLASSES, pretrained=PRETRAINED).to(DEVICE)
# DataParallel skipped: TripletAttention permute ops incompatible with DP scatter

optimizer_s2  = torch.optim.Adam(model_s2.parameters(), lr=LR)
scheduler_s2  = torch.optim.lr_scheduler.StepLR(
                    optimizer_s2, step_size=LR_STEP, gamma=LR_GAMMA)

if CLASS_WEIGHTS is not None:
    criterion_s2 = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))
    print('[S2] CrossEntropyLoss with class_weights')
else:
    criterion_s2 = nn.CrossEntropyLoss()
    print('[S2] CrossEntropyLoss (no weighting)')

print('=== Scenario 2 — Augmented (Rotation + ColorJitter + HFlip + Affine) ===')
print(f'Total parameters: {count_params(model_s2):,}')

history_s2 = train_model(
    model_s2, train_loader2, val_loader2,
    optimizer_s2, scheduler_s2, criterion_s2,
    DEVICE, EPOCHS,
    ckpt_path='/kaggle/working/best_model_s2.pth',
)
print('Training complete.')

In [ ]:
# ── Scenario 2: Classification Report ──────────────────────────────────────
_, s2_acc, s2_preds, s2_labels = evaluate(
    model_s2, test_loader2, criterion_s2, DEVICE)

s2_f1  = f1_score(s2_labels, s2_preds, average='macro')
s2_rec = recall_score(s2_labels, s2_preds, average='macro')

print('\n=== Scenario 2 — Test Metrics ===')
print(f'Accuracy     : {s2_acc:.4f}')
print(f'F1-Score (M) : {s2_f1:.4f}')
print(f'Recall   (M) : {s2_rec:.4f}')
print(f'Parameters   : {count_params(model_s2):,}')
print('\n--- Per-class Classification Report ---')
print(classification_report(s2_labels, s2_preds,
      target_names=class_names, digits=4))


In [ ]:
# ── Scenario 2: Confusion Matrix ────────────────────────────────────────────
AUG_CM_PATH = '/kaggle/working/aug_confusion_matrix.png'

cm2 = confusion_matrix(s2_labels, s2_preds)
fig, ax = plt.subplots(figsize=(max(6, NUM_CLASSES), max(5, NUM_CLASSES - 1)))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix — Scenario 2 (Augmented)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True',      fontsize=12)
plt.tight_layout()
plt.savefig(AUG_CM_PATH, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {AUG_CM_PATH}')


In [ ]:
# ── Scenario 2: Learning Curve ──────────────────────────────────────────────
AUG_LC_PATH = '/kaggle/working/aug_learning_curve.png'

ep2 = range(1, len(history_s2['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep2, history_s2['train_loss'], label='Train Loss', color='seagreen')
axes[0].plot(ep2, history_s2['val_loss'],   label='Val Loss',   color='darkorange')
axes[0].set_title('Loss — Scenario 2 (Augmented)', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep2, history_s2['train_acc'], label='Train Acc', color='seagreen')
axes[1].plot(ep2, history_s2['val_acc'],   label='Val Acc',   color='darkorange')
axes[1].set_title('Accuracy — Scenario 2 (Augmented)', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('RTR_Lite_MobileNet | Scenario 2 — With Augmentation',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(AUG_LC_PATH, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {AUG_LC_PATH}')


In [ ]:
# ── Scenario 2: metrics dict (for Excel export) ─────────────────────────────
aug_metrics = {
    'model_name'   : 'RTR_Lite_MobileNet',
    'accuracy'     : s2_acc,
    'f1_macro'     : s2_f1,
    'recall_macro' : s2_rec,
    'total_params' : count_params(model_s2),
}
print('aug_metrics:', aug_metrics)


---
## Export Results to Excel
Tạo `results_summary.xlsx` với 2 sheet, ảnh nhúng trực tiếp.


In [ ]:
# ── Install Excel deps ───────────────────────────────────────────────────────
!pip install -q openpyxl Pillow


In [ ]:
# ── Export results_summary.xlsx ─────────────────────────────────────────────
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

EXCEL_PATH     = '/kaggle/working/results_summary.xlsx'
PAPER_CITATION = 'Duhan et al. (2025)'
HEADERS        = ['STT', 'Ten Model', 'Accuracy', 'F1-score',
                  'Recall', 'Learning Curve', 'Confusion Matrix', 'Parameters']
COL_WIDTHS     = [6, 36, 12, 12, 12, 38, 38, 18]


def _header(ws):
    fill   = PatternFill('solid', fgColor='1F4E79')
    font   = Font(bold=True, color='FFFFFF', size=11)
    bdr    = Border(bottom=Side(style='medium', color='FFFFFF'),
                    right=Side(style='thin',   color='FFFFFF'))
    for col, h in enumerate(HEADERS, 1):
        c = ws.cell(row=1, column=col, value=h)
        c.fill, c.font, c.border = fill, font, bdr
        c.alignment = Alignment(horizontal='center', vertical='center',
                                wrap_text=True)


def _data_row(ws, row, stt, m, lc, cm):
    data = [stt,
            f"{m['model_name']} ({PAPER_CITATION})",
            round(m['accuracy'],     4),
            round(m['f1_macro'],     4),
            round(m['recall_macro'], 4),
            '', '',
            f"{m['total_params']:,}"]
    for col, v in enumerate(data, 1):
        ws.cell(row=row, column=col, value=v).alignment = Alignment(
            horizontal='center', vertical='center', wrap_text=True)
    ws.row_dimensions[row].height = 200
    for path, col in [(lc, 'F'), (cm, 'G')]:
        if os.path.exists(path):
            img = XLImage(path)
            img.width, img.height = 280, 190
            ws.add_image(img, f'{col}{row}')


def _sheet(wb, name, m, lc, cm):
    ws = wb.create_sheet(title=name)
    for col, w in enumerate(COL_WIDTHS, 1):
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[1].height = 30
    _header(ws)
    _data_row(ws, 2, 1, m, lc, cm)


wb = Workbook()
wb.remove(wb.active)

_sheet(wb, 'Baseline',  baseline_metrics, BASELINE_LC_PATH, BASELINE_CM_PATH)
_sheet(wb, 'Augmented', aug_metrics,      AUG_LC_PATH,      AUG_CM_PATH)

wb.save(EXCEL_PATH)
print(f'[Excel] Saved: {EXCEL_PATH}')
print(f'  Baseline  -> Acc={baseline_metrics["accuracy"]:.4f}'
      f' F1={baseline_metrics["f1_macro"]:.4f}'
      f' Recall={baseline_metrics["recall_macro"]:.4f}'
      f' Params={baseline_metrics["total_params"]:,}')
print(f'  Augmented -> Acc={aug_metrics["accuracy"]:.4f}'
      f' F1={aug_metrics["f1_macro"]:.4f}'
      f' Recall={aug_metrics["recall_macro"]:.4f}'
      f' Params={aug_metrics["total_params"]:,}')


---
## Output Artifacts
```
/kaggle/working/
├── results_summary.xlsx          ← 2 sheets (Baseline + Augmented), ảnh nhúng
├── baseline_learning_curve.png
├── baseline_confusion_matrix.png
├── aug_learning_curve.png
└── aug_confusion_matrix.png
```
Tải `results_summary.xlsx` từ tab **Output** trên Kaggle.


---
## A2 — Multi-seed robustness (mean ± std)

Đo độ ổn định của **mô hình gốc** qua nhiều seed để lấy `mean ± std` (đối chứng công bằng với multi-seed của mô hình đề xuất). Chạy **đúng 1 lần** — vòng lặp bên trong tự train `3 seed × 2 kịch bản = 6` lần. Xuất `orig_results_multiseed.csv` (thô) + `orig_results_multiseed_summary.csv` (mean ± std). Dùng checkpoint riêng `orig_ms_*.pth`, không đè `best_model_s1/s2.pth`.


In [ ]:
# ===== A2 — Multi-seed robustness (mean ± std cho MÔ HÌNH GỐC) =====
# Chạy ĐÚNG 1 LẦN sau khi Run All. Vòng lặp bên trong tự train 3 seed × 2 kịch bản.
# Dùng checkpoint riêng orig_ms_*.pth → KHÔNG đè best_model_s1/s2.pth chính thức.
# Hết giờ GPU: sửa RUN_SCENARIOS = ['S2'] (chỉ kịch bản augmented headline).
import random, pandas as pd
from sklearn.metrics import precision_score

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

SEEDS         = [42, 1, 2]
RUN_SCENARIOS = ['S1', 'S2']
_loaders = {
    'S1': (train_loader1, val_loader1, test_loader1),
    'S2': (train_loader2, val_loader2, test_loader2),
}

ms_rows = []
for seed in SEEDS:
    for sc in RUN_SCENARIOS:
        tr_ld, vl_ld, te_ld = _loaders[sc]

        set_seed(seed)                        # seed cho khởi tạo trọng số
        m = RTR_Lite_MobileNet(NUM_CLASSES, pretrained=PRETRAINED).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=LR)
        sch = torch.optim.lr_scheduler.StepLR(opt, step_size=LR_STEP, gamma=LR_GAMMA)
        crit = (nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))
                if CLASS_WEIGHTS is not None else nn.CrossEntropyLoss())

        set_seed(seed)                        # seed lại cho thứ tự shuffle dữ liệu
        _ = train_model(m, tr_ld, vl_ld, opt, sch, crit, DEVICE, EPOCHS,
                        ckpt_path=f'orig_ms_{sc}_seed{seed}.pth')

        _, acc, preds, labs = evaluate(m, te_ld, crit, DEVICE)
        f1   = f1_score(labs, preds, average='macro')
        rec  = recall_score(labs, preds, average='macro')
        prec = precision_score(labs, preds, average='macro')
        ms_rows.append({'seed': seed, 'scenario': sc, 'acc': acc, 'f1': f1,
                        'recall': rec, 'precision': prec, 'params': count_params(m)})
        print(f'[MS] seed={seed} {sc} | acc={acc:.4f} P={prec:.4f} R={rec:.4f} f1={f1:.4f}')

        del m, opt, sch; torch.cuda.empty_cache(); gc.collect()

ms_df = pd.DataFrame(ms_rows)
ms_df.to_csv('orig_results_multiseed.csv', index=False)

summary = ms_df.groupby('scenario').agg(
    n_runs=('seed', 'count'),
    acc_mean=('acc', 'mean'),             acc_std=('acc', 'std'),
    precision_mean=('precision', 'mean'), precision_std=('precision', 'std'),
    recall_mean=('recall', 'mean'),       recall_std=('recall', 'std'),
    f1_mean=('f1', 'mean'),               f1_std=('f1', 'std'),
).reset_index()
summary.to_csv('orig_results_multiseed_summary.csv', index=False)

print('\n===== MULTI-SEED SUMMARY (mô hình gốc) =====')
print(summary.to_string(index=False))
print('\nSaved: orig_results_multiseed.csv + orig_results_multiseed_summary.csv')
